# Intel Image Classification using Neural Networks, CNNs and Transfer Learning
### Delivered by
- Mohamed Abdelmohsen Soliman

## **Required Libraries**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns

import tqdm as notebook_tqdm
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, f1_score, classification_report
from sklearn.preprocessing import LabelEncoder, StandardScaler

from tqdm.auto import tqdm
import torch
import torch.nn as nn
import torchvision
from torch.utils.data import TensorDataset, DataLoader, Subset, random_split
from torchvision import datasets, transforms


## **Python Version Used**

In [ ]:
import sys
print(sys.version)

### **Libraries versions**
I have attached the **requirements.txt** in the repo as well

In [ ]:
## Libraries version check
print(f"Pandas version: {pd.__version__}")
print(f"Numpy version: {np.__version__}")
print(f"Matplotlib version: {matplotlib.__version__}")
print(f"Seaborn version: {sns.__version__}")
print(f"PyTorch version: {torch.__version__}")
print(f"Scikit-learn version: {sklearn.__version__}")
print(f"torchvision version: {torchvision.__version__}")

### To make a model reproducible, we must fix the seed of any randomness!
##### Additionally, we save each model with its best weights to be able to reload it anytime.
I am doing that after traininig of each model below.

In [ ]:
import random
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

### Reading the Dataset and view it

## 📂 Dataset: Intel Image Classification

* **Dataset Domain:** Natural landscapes and urban scenes inot 6 classes (Buildings, Forest, Glacier, Mountain, Sea, Street).
<!-- * **Model Domain:** High overlap with **ImageNet** (used in Task 3), as it contains similar outdoor and architectural categories. -->

**Data Split & Structure**

* **`seg_train`**: ~14,000 labeled images for training all models.
* **`seg_test`**: ~3,000 labeled images for validation and metric reporting (Acc, Recall, etc.).
* **`seg_pred`**: ~7,000 unlabeled images (Optional for final blind testing, wont be used here).

**Technical Strategy**

* **Lazy Loading:** Implemented via DataLoaders to manage memory efficiency.
* **Sampling:** Data will be sampled to a smaller percentage to ensure fast training without introducing class bias.

---

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("puneet6060/intel-image-classification")

print("Path to dataset files:", path)

In [ ]:
# path = 'Datasets/IntelData'
TrainDataPath = f"{path}/seg_train/seg_train"
TestDataPath = f"{path}/seg_test/seg_test"

## Let us visualize the data and see its info

In [ ]:
import matplotlib.pyplot as plt
import os
import random
from PIL import Image

# here we get all class names from folder names
classes = os.listdir(TrainDataPath)

plt.figure(figsize=(8, 8))

for i in range(4):
    # first we pick a random class
    random_class = random.choice(classes)
    class_path = os.path.join(TrainDataPath, random_class)

    # the we pick a random image from that class
    random_img_name = random.choice(os.listdir(class_path))
    img_path = os.path.join(class_path, random_img_name)

#load it and plot
    img = Image.open(img_path)
    plt.subplot(2, 2, i + 1)
    plt.imshow(img)
    plt.title(f"Class: {random_class}")
    plt.axis('off')

plt.tight_layout()
plt.show()

### Checking the data balance

In [ ]:
import os

counts = {}
for label in os.listdir(TrainDataPath):
    class_path = os.path.join(TrainDataPath, label) # we check if the path is a directory to avoid any issues with hidden files or non-directory items
    if os.path.isdir(class_path):
        counts[label] = len(os.listdir(class_path)) # we count the number of images in each class by listing the files in the class directory and getting their length

counts = dict(sorted(counts.items(), key=lambda x: x[1], reverse=True)) # here we sort the counts in descending order

for label, count in counts.items():
    print(f"{label}: {count} images")

fig, ax = plt.subplots()
ax.bar(counts.keys(), counts.values())
ax.set_title("Data Imbalance Check")
plt.tight_layout()
plt.show()

To fit your current task with the **Intel Image Classification** dataset, you need to update the description to reflect image data rather than tabular features.

Here is the updated version for your notebook:

#### Now we have ~17K Images with 3 Color Channels (RGB) to predict 6 different classes:

* **Buildings** - 0
* **Forest** - 1
* **Glacier** - 2
* **Mountain** - 3
* **Sea** - 4
* **Street** - 5

---

### choose the gpu as the target device

In [ ]:
device = torch.device("cuda")

### The data is perfectly balanced. We are good to go and start loading the data & learning

## **Data Preprocessing & Augmentation**

In [ ]:

transform = transforms.Compose([ # here we are transforming the images using flipping, rotation, resizing and converting to tensor. This helps in data augmentation and preparing the images for training.
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.Resize((150, 150)), #  we resize all images to a consistent size (150x150) to ensure uniformity
    transforms.ToTensor()
])

# the Lazy Loading part via ImageFolder from torch
full_train_dataset = datasets.ImageFolder(root=TrainDataPath, transform=transform)
test_dataset = datasets.ImageFolder(root=TestDataPath, transform=transform)

# We take 100% of the indices randomly since the dataset is already small
num_train = len(full_train_dataset)
indices = list(range(num_train))
np.random.shuffle(indices)
train_idx = indices[:] # we take all indices since the dataset is already small, but we shuffle them to ensure randomness

train_subset = Subset(full_train_dataset, train_idx)

# This is the Dataloaders for training and testing
train_dataset, val_dataset = random_split(train_subset, [0.9, 0.1]) # we only take 10% for the val set

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print(f"Using {len(train_dataset)} images for training.")
print(f"Using {len(val_dataset)} images for training.")

## **Simple MLP**

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class SimpleMLP(nn.Module):
    def __init__(self):
        super(SimpleMLP, self).__init__()
        # flatten: 150*150*3 = 67500
        self.fc1 = nn.Linear(150 * 150 * 3, 512)
        self.fc2 = nn.Linear(512, 256)
        self.fc3 = nn.Linear(256, 6) # 6 Output classes which are : ['buildings', 'forest', 'glacier', 'mountain', 'sea', 'street']

    def forward(self, x):
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

model_mlp = SimpleMLP()

In [ ]:
model_mlp.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model_mlp.parameters(), lr=0.005)

### Ensuring that gpu is selected

In [ ]:
print(f"Using device: {device}")

### Sanity Check Function, will be called to ensure the Golden Rules of learning

##### We are letting the model memorize the subset to check if our piepline is safe to go with

In [ ]:
import torch
from torch.utils.data import DataLoader, Subset
from sklearn.metrics import accuracy_score

def SanityCheck(sanityModel, epochs): # a sanity check func that takes the model, and number of epochs as arguments
    # picking a tiny subset (100 samples) from the dataset
    sanity_indices = range(100)
    sanity_subset = Subset(train_dataset, sanity_indices)
    sanity_loader = DataLoader(sanity_subset, batch_size=20, shuffle=True)

    print("=" * 60)
    print("SANITY CHECK: Overfitting on 100 Intel Images")
    print("=" * 60)

    sanity_model = sanityModel

    sanity_optimizer = torch.optim.Adam(sanity_model.parameters(), lr=0.0008)
    sanity_criterion = nn.CrossEntropyLoss()

    for epoch in range(epochs): # run for the number of params-epochs
        sanity_model.train()
        running_loss = 0.0

        for images, labels in sanity_loader:
            images, labels = images.to(device), labels.to(device)

            sanity_optimizer.zero_grad()
            outputs = sanity_model(images)
            loss = sanity_criterion(outputs, labels)
            loss.backward()
            sanity_optimizer.step()

            running_loss += loss.item()

        # Check progress every 10 epochs
        if (epoch + 1) % 10 == 0:
            sanity_model.eval()
            all_preds = []
            all_labels = []

            with torch.no_grad():
                for images, labels in sanity_loader:
                    images = images.to(device)
                    logits = sanity_model(images)
                    preds = torch.argmax(logits, dim=1)

                    all_preds.extend(preds.cpu().numpy())
                    all_labels.extend(labels.numpy())

            acc = accuracy_score(all_labels, all_preds)
            print(f"Epoch {epoch+1:02d} | Loss: {running_loss/len(sanity_loader):.4f} | Acc: {acc:.4f}")

            if acc == 1.0:
                print("\n✅ SUCCESS: Model memorized the subset. The pipeline is healthy!")
                break

### Here we are running the sanity check for the MLP

In [ ]:
SanityCheck(SimpleMLP().to(device), 200)

In [ ]:
from tqdm.notebook import tqdm

def train(model, optimizer, criterion, epochs, device, train_loader, val_loader):

  # Dictionary to store metrics for plotting later
  history = {
      'train_loss': [], 'train_acc': [],
      'val_loss': [], 'val_acc': []
  }

  for epoch in range(epochs):

      model.train()
      running_loss = 0.0
      correct = 0
      total = 0

      # this is the tqdm progress bar
      loop = tqdm(train_loader, desc=f"Epoch [{epoch+1}/{epochs}]", leave=True)

      for images, labels in loop: # looping over the image
          images, labels = images.to(device), labels.to(device)

          optimizer.zero_grad()
          outputs = model(images)
          loss = criterion(outputs, labels)
          loss.backward()
          optimizer.step()

          # Metrics incrementing and storing
          running_loss += loss.item()
          _, predicted = torch.max(outputs.data, 1)
          total += labels.size(0)
          correct += (predicted == labels).sum().item()

          # updating the progress bar with live stats
          loop.set_postfix(loss=loss.item(), acc=100.0 * correct / total)

      # here we calculate Epoch Training Metrics and append it into the arrays
      train_loss = running_loss / len(train_loader)
      train_acc = 100 * correct / total
      history['train_loss'].append(train_loss)
      history['train_acc'].append(train_acc)

      model.eval()
      val_running_loss = 0.0
      val_correct = 0
      val_total = 0

      with torch.no_grad(): # No gradients needed for validation
          for images, labels in val_loader:
              images, labels = images.to(device), labels.to(device)
              outputs = model(images)
              loss = criterion(outputs, labels)

              val_running_loss += loss.item()
              _, predicted = torch.max(outputs.data, 1)
              val_total += labels.size(0)
              val_correct += (predicted == labels).sum().item()

      # here we calulcate Epoch Validation Metrics
      val_loss = val_running_loss / len(val_loader)
      val_acc = 100 * val_correct / val_total
      history['val_loss'].append(val_loss)
      history['val_acc'].append(val_acc)

      # Summarizing the metrics
      print(f"Summary Epoch {epoch+1}: "
            f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}% | "
            f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%")

  return history

In [ ]:
def evaluate_model(model, data_loader, criterion, device):
    model.eval()
    total_loss = 0.0
    # here we are initializing lists to append the preds and labels
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for xb, yb in data_loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            loss = criterion(logits, yb)

            total_loss += loss.item() * xb.size(0)
            preds = torch.argmax(logits, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(yb.cpu().numpy())

    avg_loss = total_loss / len(data_loader.dataset)
    acc = accuracy_score(all_labels, all_preds)
    return avg_loss, acc, np.array(all_labels), np.array(all_preds)

In [ ]:
def plot_metrics(train_losses, val_losses, train_accuracies, val_accuracies): # a plot function that takes the history arrays and plots them
    plt.figure(figsize=(10, 5))
    plt.plot(train_losses, label="Train Loss")
    plt.plot(val_losses, label="Validation Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Training vs Validation Loss")
    plt.legend()
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(10, 5))
    plt.plot(train_accuracies, label="Train Accuracy")
    plt.plot(val_accuracies, label="Validation Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title("Training vs Validation Accuracy")
    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
def evaluate_on_test(model, test_loader, criterion, device):
    model.eval()

    total_loss = 0.0
    all_preds = []
    all_labels = []

    # the same tqdm progress bar for evals
    loop = tqdm(test_loader, desc="Evaluating", leave=False)

    with torch.no_grad(): # here we are evaluating the model on the test set, calculating the loss and predictions for each batch, and appending them to the respective lists.
        for xb, yb in loop:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            loss = criterion(logits, yb)

            total_loss += loss.item() * xb.size(0)
            preds = torch.argmax(logits, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(yb.cpu().numpy())

    # here i am calculating the average loss, accuracy, precision, recall, and f1-score for both macro and weighted averages.
    avg_loss = total_loss / len(test_loader.dataset)
    accuracy = accuracy_score(all_labels, all_preds)
    precision_macro = precision_score(all_labels, all_preds, average="macro")
    recall_macro = recall_score(all_labels, all_preds, average="macro")
    f1_macro = f1_score(all_labels, all_preds, average="macro")
    precision_weighted = precision_score(all_labels, all_preds, average="weighted")
    recall_weighted = recall_score(all_labels, all_preds, average="weighted")
    f1_weighted = f1_score(all_labels, all_preds, average="weighted")

    print("=" * 60)
    print("TEST SET EVALUATION")
    print("=" * 60)
    print(f"Test Loss: {avg_loss:.4f}")
    print(f"Test Accuracy: {accuracy:.4f}")
    print("-" * 40)
    print(f"Macro Precision: {precision_macro:.4f}")
    print(f"Macro Recall: {recall_macro:.4f}")
    print(f"Macro F1-score: {f1_macro:.4f}")
    print("-" * 40)
    print(f"Weighted Precision: {precision_weighted:.4f}")
    print(f"Weighted Recall: {recall_weighted:.4f}")
    print(f"Weighted F1-score: {f1_weighted:.4f}")
    print("\nClassification Report:\n")
    print(classification_report(all_labels, all_preds))

    # Visualizing the results through the confusion matrix
    cm = confusion_matrix(all_labels, all_preds)
    plt.figure(figsize=(8,6))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title("Confusion Matrix - Test Set")
    plt.show()

In [ ]:
history = train(model_mlp, optimizer, criterion, 15, device, train_loader, val_loader)

In [ ]:
loss, acc , _, _ = evaluate_model(model_mlp, val_loader, criterion, device)
print(f"Val Loss = {loss}")
print(f"Val Acc = {acc}")

In [ ]:
evaluate_on_test(model_mlp, test_loader, criterion, device)

In [ ]:
import matplotlib.pyplot as plt

plot_metrics(history['train_loss'], history['val_loss'], history['train_acc'], history['val_acc'])

In [ ]:
torch.save(model_mlp.state_dict(), 'SimpleMLP-Intel.pth')
print("Model state dict saved successfully!")

## The Second Model (CNN)!
This is our first spatial model. Unlike the MLP, this CNN uses convolutional layers to extract features from images, followed by a pooling layer to reduce dimensionality, and finally a fully connected layer to map those features to our 6 target classes.

In [ ]:
class CNN(nn.Module): # switched from MLP to CNN
    def __init__(self, num_classes, mode="plain"):
        super(CNN, self).__init__()

        layers = []

        # Convolutional layers to 'see' patterns in the images
        # first layer, we picked 3 channels since they are rgb images
        layers.append(nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, stride=1, padding=1))

        if mode == "dropout_bn":
            layers.append(nn.BatchNorm2d(16))

        layers.append(nn.ReLU())
        layers.append(nn.MaxPool2d(kernel_size=2, stride=2)) # Reduces image size by half

        layers.append(nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, stride=1, padding=1))

        if mode == "dropout_bn":
            layers.append(nn.BatchNorm2d(32))

        layers.append(nn.ReLU())
        layers.append(nn.MaxPool2d(kernel_size=2, stride=2))

        self.features = nn.Sequential(*layers)

        # Classifier: The "Linear" part that makes the final decision
        # Note: 'flatten_size' depends on your input image dimensions (e.g., 64x64 or 128x128)
        classifier_layers = [
            nn.Flatten(),
            nn.Linear(32 * 37 * 37, 128), # Update the first number based on your image size!
            nn.ReLU()
        ]

        if mode in ["dropout", "dropout_bn"]:
            classifier_layers.append(nn.Dropout(0.3))

        classifier_layers.append(nn.Linear(128, num_classes))

        self.classifier = nn.Sequential(*classifier_layers)

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x



### Lets run for sanity check and check if the pipeline is healthy to go with

In [ ]:
SanityCheck(CNN(6).to(device), 100)

# Simplest CNN

In [ ]:
SimpleCnnModel = CNN(6, "plain")
SimpleCnnModel.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(SimpleCnnModel.parameters(), lr=0.0005)

print(SimpleCnnModel)

In [ ]:
print("Some filters before training:")

filters_before = SimpleCnnModel.features[0].weight.clone().detach()
fig, ax = plt.subplots(2,2)
for i in range(2):
  for j in range (2):
    ax[i][j].imshow(filters_before[i][j].cpu().detach().numpy(), cmap='gray')
plt.show()

In [ ]:
hist = train(SimpleCnnModel, optimizer, criterion, 15, device, train_loader, val_loader)

In [ ]:
val_loss, val_acc, _, _ = evaluate_model(SimpleCnnModel, val_loader, criterion, device)
print("Val_Loss = ", val_loss)
print("Val_Acc = ", val_acc)

In [ ]:
evaluate_on_test(SimpleCnnModel, test_loader, criterion, device)

In [ ]:
plot_metrics(hist['train_loss'], hist['val_loss'], hist['train_acc'], hist['val_acc'])

In [ ]:
torch.save(SimpleCnnModel.state_dict(), 'CNNSimple_Intel.pth')
print("Model state dict saved successfully!")

In [ ]:
# after training
filters_after = SimpleCnnModel.features[0].weight.clone().detach()

In [ ]:
print("Some filters after training:")
fig, ax = plt.subplots(2,2)
for i in range(2):
  for j in range (2):
    ax[i][j].imshow(filters_after[i][j].cpu().detach().numpy(), cmap='gray')
plt.show()

# CNN With Dropout

In [ ]:
DropoutCnnModel = CNN(6, "dropout")
DropoutCnnModel.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(DropoutCnnModel.parameters(), lr=0.0005)

print(DropoutCnnModel)

In [ ]:
print("Some filters before training:")
filters_before = DropoutCnnModel.features[0].weight.clone().detach()
fig, ax = plt.subplots(2,2)
for i in range(2):
  for j in range (2):
    ax[i][j].imshow(filters_before[i][j].cpu().detach().numpy(), cmap='gray')
plt.show()

In [ ]:
hist = train(DropoutCnnModel, optimizer, criterion, 15, device, train_loader, val_loader)

In [ ]:
val_loss, val_acc, _, _ = evaluate_model(DropoutCnnModel, val_loader, criterion, device)
print("Val_Loss = ", val_loss)
print("Val_Acc = ", val_acc)

In [ ]:
evaluate_on_test(DropoutCnnModel, test_loader, criterion, device)

In [ ]:
plot_metrics(hist['train_loss'], hist['val_loss'], hist['train_acc'], hist['val_acc'])

In [ ]:
torch.save(DropoutCnnModel.state_dict(), 'CNNComplex_Intel.pth')
print("Model state dict saved successfully!")

In [ ]:
# after training
filters_after = DropoutCnnModel.features[0].weight.clone().detach()

In [ ]:
print("Some filters after training:")
fig, ax = plt.subplots(2,2)
for i in range(2):
  for j in range (2):
    ax[i][j].imshow(filters_after[i][j].cpu().detach().numpy(), cmap='gray')
plt.show()

# CNN With Dropout, Weight Decay, BatchNorm and Lr Schedule

In [ ]:
BatchNormDropoutCnnModel = CNN(6, "dropout_bn")
BatchNormDropoutCnnModel.to(device)

criterion = nn.CrossEntropyLoss()

# weight decay helps regularization
optimizer = torch.optim.Adam(
    BatchNormDropoutCnnModel.parameters(),
    lr=0.0005,
    weight_decay=1e-4
)

print(BatchNormDropoutCnnModel)

In [ ]:
print("Some filters before training:")

filters_before = BatchNormDropoutCnnModel.features[0].weight.clone().detach()
fig, ax = plt.subplots(2,2)
for i in range(2):
  for j in range (2):
    ax[i][j].imshow(filters_before[i][j].cpu().detach().numpy(), cmap='gray')
plt.show()

In [ ]:
hist = train(BatchNormDropoutCnnModel, optimizer, criterion, 15, device, train_loader, val_loader)

In [ ]:
val_loss, val_acc, _, _ = evaluate_model(BatchNormDropoutCnnModel, val_loader, criterion, device)
print("Val_Loss = ", val_loss)
print("Val_Acc = ", val_acc)

In [ ]:
evaluate_on_test(BatchNormDropoutCnnModel, test_loader, criterion, device)

In [ ]:
plot_metrics(hist['train_loss'], hist['val_loss'], hist['train_acc'], hist['val_acc'])

In [ ]:
torch.save(BatchNormDropoutCnnModel.state_dict(), 'CNNRegularized_Intel.pth')
print("Model state dict saved successfully!")

In [ ]:
# after training
filters_after = BatchNormDropoutCnnModel.features[0].weight.clone().detach()

In [ ]:
print("Some filters after training:")

fig, ax = plt.subplots(2,2)
for i in range(2):
  for j in range (2):
    ax[i][j].imshow(filters_after[i][j].cpu().detach().numpy(), cmap='gray')
plt.show()

### Here we are plotting some of the filters before training

The last model, which is the most complex one and it is a pretrained model. I will load the ResNet18 architecture, which has been pretrained on millions of images from the ImageNet database.

By using Transfer Learning, we can take the "knowledge" this model already has like recognizing textures and complex shapes and fine-tune or freeze the last layers and retrain them.

In [ ]:
import torchvision.models as models
import torch.nn as nn
import torch

def build_resnet(mode="fc_only", num_classes=6, device="cuda", lr = 0.001):

    # load pretrained resnet18 trained on imagenet
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

    # replace the last fully connected layer
    # original resnet outputs 1000 classes but our dataset has 6
    num_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Linear(num_features, 256),
        nn.ReLU(),
        nn.Dropout(0.25),
        nn.Linear(256, num_classes)
    )

    # transfer learning scenarios

    if mode == "fc_only": # scenarion a

        # freeze the whole network first
        for param in model.parameters():
            param.requires_grad = False

        # train only the classifier layer
        for param in model.fc.parameters():
            param.requires_grad = True


    elif mode == "finetune_last": # scenario b

        # freeze everything first
        for param in model.parameters():
            param.requires_grad = False # here we do the freezing


        # these layers learn higher level features
        for param in model.layer4.parameters():
            param.requires_grad = True # here we unfreeze the last convolution block

        # also train the classifier
        for param in model.fc.parameters():
            param.requires_grad = True


    elif mode == "finetune_all": #scenario c

        # train the entire network
        for param in model.parameters():    # this allows the model to fully adapt to the dataset
            param.requires_grad = True


    model = model.to(device)

    optimizer = torch.optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr # learning rat of 0.001
    )

    criterion = nn.CrossEntropyLoss()

    return model, optimizer, criterion

In [ ]:
import matplotlib.pyplot as plt

def plot_transfer_metrics(hist_fc, hist_last, hist_all):

    histories = [
        ("FC Only", hist_fc),
        ("Finetune Last", hist_last),
        ("Finetune All", hist_all)
    ]

    fig, axes = plt.subplots(3, 2, figsize=(12, 12))

    for i, (title, hist) in enumerate(histories):

        # loss plot
        axes[i,0].plot(hist["train_loss"], label="train")
        axes[i,0].plot(hist["val_loss"], label="val")
        axes[i,0].set_title(f"{title} Loss")
        axes[i,0].set_xlabel("epoch")
        axes[i,0].set_ylabel("loss")
        axes[i,0].legend()

        # accuracy plot
        axes[i,1].plot(hist["train_acc"], label="train")
        axes[i,1].plot(hist["val_acc"], label="val")
        axes[i,1].set_title(f"{title} Accuracy")
        axes[i,1].set_xlabel("epoch")
        axes[i,1].set_ylabel("accuracy")
        axes[i,1].legend()

    plt.tight_layout()
    plt.show()

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

In [ ]:
model_fc_only, optimizerA, criterionA = build_resnet("fc_only", 6, device, 0.001)

In [ ]:
model_finetune_last, optimizerB, criterionB = build_resnet("finetune_last", 6, device, 0.001)

In [ ]:
model_finetune_all, optimizerC, criterionC = build_resnet("finetune_all", 6, device, 0.0008)

In [ ]:
histA = train(model_fc_only, optimizerA, criterionA, 10, device, train_loader, val_loader)

In [ ]:
histB = train(model_finetune_last, optimizerB, criterionB, 10, device, train_loader, val_loader)

In [ ]:
histC = train(model_finetune_all, optimizerC, criterionC, 10, device, train_loader, val_loader)

In [ ]:
# i am re-evaluating to get the "Clean" metrics and logging them in a nice format in the end.
scenarios = {
    "FC Only": (model_fc_only, criterionA),
    "Finetune Last Layers": (model_finetune_last, criterionB),
    "Finetune All": (model_finetune_all, criterionC)
}

for name, (model, criterion) in scenarios.items():

    train_loss_final, train_acc_final, _, _ = evaluate_model(model, train_loader, criterion, device)
    val_loss_final, val_acc_final, _, _ = evaluate_model(model, val_loader, criterion, device)

    print("\n")
    print("="*50)
    print(f"{name}")
    print("="*50)
    print(f"{'METRIC':<20} | {'VALUE':<10}")
    print("="*50)
    print(f"{'Final Train Loss':<20} | {train_loss_final:.4f}")
    print(f"{'Final Train Acc':<20} | {train_acc_final*100:.2f}%")
    print("-" * 50)
    print(f"{'Final Val Loss':<20} | {val_loss_final:.4f}")
    print(f"{'Final Val Acc':<20} | {val_acc_final*100:.2f}%")
    print("-" * 50)
    print(f"{'Generalization Gap':<20} | {np.abs((val_acc_final - train_acc_final))*100:.2f}%")
    print("="*50)

In [ ]:
plot_transfer_metrics(histA, histB, histC)

In [ ]:
evaluate_on_test(model_fc_only, test_loader, criterionA, device)

In [ ]:
evaluate_on_test(model_finetune_last, test_loader, criterionB, device)

In [ ]:
evaluate_on_test(model_finetune_all, test_loader, criterionC, device)

and it yielded the most decent confusion matrix out of the 3!

### In this deep learning task, I implemented **three neural network architectures** with increasing complexity to evaluate their effectiveness on the Intel Image Classification problem. To align with the golden standards, i have started with a sanity check by overfitting a small subset of the data to see the results. Then, I have trained and MLP on the Intel data and yielded a very bad results, almost 50% accuracy. The second model was a Convolutional Neural Netwok and it yielded an accuracy of 75%. The Last model was a pretrained model ResNet-18 and it yielded a training accuracy of 90% and 88% val accuracy with a very decent confusion matrix.

### **Reproducibility** was ensured by fixing random seeds and saving trained model weights for any usage in the future. This experiment demonstrates the importance of progressive model designing and a proper evaluation pipeline for any deep nn projects or tasks

# Thank You
### Delivered by
- Mohamed Abdelmohsen Soliman